# 🔎 DataScout — Conversational Data Analysis Agent

**Track:** Concierge Agents
**Base course:** Kaggle × Google — 5-Day AI Agents: Intensive Vibe Coding Course

DataScout is an agent you hand a CSV to. Using Gemini **function calling**, it decides
which tool to run at each step: exploring the dataset, detecting outliers, running
correlations, plotting, or summarizing in natural language. It's not a fixed pipeline —
the model reasons about what to do based on what you ask.

### 4 Key Concepts demonstrated (course requirement: 3+)
| Key Concept | Where | How |
|---|---|---|
| **Security features** | Code | `modify_data()` requires explicit human confirmation before any action that changes the dataset — no silent mutations. Argument validation (added after a STRIDE-style review) also prevents crashes from hallucinated column names. |
| **Agent Skills** | Code | The agent's instructions live in a separate `SKILL.md` file (in `.agent/skills/`), loaded at runtime — not hardcoded in Python, mirroring Antigravity's native skill pattern. |
| **Deployability** | Video / Code | Gemini API key is never hardcoded — read from Colab Secrets (`google.colab.userdata`), so this notebook is safe to share and run publicly. |
| **Antigravity** *(bonus)* | Video | Google's Antigravity IDE was used to run a real STRIDE-style threat model on `modify_data()`, documented in `SECURITY.md`, and the finding was fixed in code. |

Additional concepts touched on: real tool use via Gemini function calling, and context
engineering through `chat_history` and `session_memory`.

---
### ⚠️ API key setup (IMPORTANT — never hardcode it)
1. Open the **Secrets** panel in Colab (🔑 icon, left sidebar).
2. Create a secret named `GEMINI_API_KEY` and paste your key from [Google AI Studio](https://aistudio.google.com/apikey).
3. Toggle "Notebook access" on for that secret.
4. Run the cells normally — the key is never written into this file.


In [ ]:
!pip install -q google-genai pandas matplotlib scipy

In [ ]:
from google.colab import userdata
from google import genai

# The key is read from Colab's Secrets system — it is never written into this notebook.
API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=API_KEY)

MODEL = "gemini-2.5-flash"  # switch to another available model in your account if needed
print("Gemini client initialized successfully. Key loaded securely.")

## 1. Upload your dataset

In [ ]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()
fname = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[fname]))
print(f"Dataset loaded: {fname} — {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

## 1.5 Load the agent's Skill (SKILL.md)

Following the course's "Agent Skills" pattern (Day 3), the agent's instructions are
NOT hardcoded in Python — they live in a separate `SKILL.md` file, loaded at runtime.
This lets the agent's behavior be versioned and edited independently from the
orchestration code, and is the foundation for future multi-agent setups (e.g. a
"cleaning agent" + this "analysis agent"), each with its own skill definition.

Upload the `SKILL.md` file included alongside this notebook (from `.agent/skills/datascout-analysis/SKILL.md`).

In [ ]:
from google.colab import files as skill_files

print("Upload the SKILL.md file:")
skill_upload = skill_files.upload()
skill_filename = list(skill_upload.keys())[0]

with open(skill_filename, "r", encoding="utf-8") as f:
    SKILL_DEFINITION = f.read()

print(f"\nSkill loaded from '{skill_filename}' ({len(SKILL_DEFINITION)} characters).")
print("First lines:\n", SKILL_DEFINITION[:200], "...")

## 2. Agent tools (function calling)

Each function is a real tool the model can decide to call. The design follows the
"10x agents" pattern from the course: the agent connects tools + reasons about which
one to use, instead of us hardcoding the flow by hand.

In [ ]:
import json
import matplotlib.pyplot as plt
from scipy import stats

# ---- Session memory (context engineering, course Day 3) ----
# This dict is the agent's short-term memory: every tool call gets logged here,
# so a future turn (or a future version with persistence) can avoid redundant
# analysis or reference prior findings ("you already checked that column").
session_memory = {
    "actions_taken": [],
    "last_summary": None,
}

def describe_data(columns: list[str] = None) -> str:
    """Returns descriptive statistics for the dataset (or specific columns)."""
    # Design note: read-only tool, safe to call without confirmation.
    subset = df[columns] if columns else df
    result = subset.describe(include="all").to_string()
    session_memory["actions_taken"].append("describe_data")
    return result

def detect_outliers(column: str, z_threshold: float = 3.0) -> str:
    """Detects outliers in a numeric column using z-score."""
    # Design note: read-only. Returns a count + threshold used, not raw row
    # indices, keeping the tool's output concise for the model to summarize.
    if column not in df.columns or not pd.api.types.is_numeric_dtype(df[column]):
        return f"Column '{column}' does not exist or is not numeric."
    z_scores = stats.zscore(df[column].dropna())
    outlier_count = int((abs(z_scores) > z_threshold).sum())
    session_memory["actions_taken"].append(f"detect_outliers:{column}")
    return f"Found {outlier_count} outliers in '{column}' (z > {z_threshold})."

def run_correlation(col_a: str, col_b: str) -> str:
    """Computes the Pearson correlation between two numeric columns."""
    # Design note: read-only. Validates both columns exist before computing,
    # since a KeyError here would surface as a raw Python traceback to the
    # model instead of a usable natural-language error.
    if col_a not in df.columns or col_b not in df.columns:
        return "One or both columns do not exist in the dataset."
    corr = df[col_a].corr(df[col_b])
    session_memory["actions_taken"].append(f"run_correlation:{col_a}-{col_b}")
    return f"Pearson correlation between '{col_a}' and '{col_b}': {corr:.3f}"

def plot_chart(column: str, chart_type: str = "histogram") -> str:
    """Generates a chart (histogram or boxplot) for a numeric column."""
    # Design note: read-only (visual output only, no dataset mutation).
    if column not in df.columns:
        return f"Column '{column}' does not exist."
    plt.figure(figsize=(6, 4))
    if chart_type == "boxplot":
        plt.boxplot(df[column].dropna())
    else:
        plt.hist(df[column].dropna(), bins=30)
    plt.title(f"{chart_type} of {column}")
    plt.show()
    session_memory["actions_taken"].append(f"plot_chart:{column}")
    return f"Chart ({chart_type}) of '{column}' generated."

def modify_data(operation: str, column: str) -> str:
    """
    SENSITIVE ACTION: modifies the dataset (e.g. dropping nulls or outliers).
    Requires human confirmation before executing (guardrail, course Day 4).
    """
    # SECURITY DESIGN: this is the ONLY function in the whole tool set that can
    # mutate `df`. Centralizing every write path through one function means
    # there is exactly one place to audit, one place the guardrail needs to
    # live, and no way for a future tool to accidentally bypass confirmation.
    # The check happens here in code, not just in the prompt/SKILL.md — so it
    # holds even if the model's reasoning were somehow wrong or manipulated.
    global df

    # SECURITY FIX (from SECURITY.md threat model, generated via Antigravity's
    # STRIDE-style review): validate arguments BEFORE touching df, so a
    # hallucinated or malformed column name from the LLM returns a clean
    # natural-language error instead of crashing with a raw KeyError/TypeError.
    if column not in df.columns:
        return f"Column '{column}' does not exist in the dataset. Available columns: {list(df.columns)}"
    if operation == "drop_outliers" and not pd.api.types.is_numeric_dtype(df[column]):
        return f"Column '{column}' is not numeric, cannot compute outliers on it."

    confirm = input(
        f"\n⚠️  The agent wants to run '{operation}' on column '{column}'. "
        f"This will modify the in-memory dataset. Confirm? (y/n): "
    )
    if confirm.strip().lower() != "y":
        return "Operation cancelled by the user. Dataset unchanged."

    if operation == "drop_na":
        before = len(df)
        df = df.dropna(subset=[column])
        session_memory["actions_taken"].append(f"modify_data:drop_na:{column}")
        return f"Dropped {before - len(df)} rows with nulls in '{column}'."
    elif operation == "drop_outliers":
        z_scores = stats.zscore(df[column].dropna())
        mask = abs(z_scores) <= 3
        before = len(df)
        df = df[df[column].isin(df[column].dropna()[mask])]
        session_memory["actions_taken"].append(f"modify_data:drop_outliers:{column}")
        return f"Dropped {before - len(df)} outliers from '{column}'."
    return "Operation not recognized."

print("Tools defined:", ["describe_data", "detect_outliers", "run_correlation", "plot_chart", "modify_data"])

## 3. Agent loop

We pass the functions to Gemini as tools. The model decides which one to call based
on the user's question; we execute the real function and return its result so the
model can compose the final natural-language answer.

In [ ]:
from google.genai import types

# Design note: tools are passed as plain Python functions, letting the
# google-genai SDK handle automatic function calling — schema inference,
# argument parsing, and execution routing are done by the library, not by us.
# This is the "vibe coding" premise in practice: we describe capabilities,
# the model decides when and how to use them.
AVAILABLE_TOOLS = {
    "describe_data": describe_data,
    "detect_outliers": detect_outliers,
    "run_correlation": run_correlation,
    "plot_chart": plot_chart,
    "modify_data": modify_data,
}

# Security design note: SKILL_DEFINITION (loaded from SKILL.md above) explicitly
# instructs the model to route every data-modifying request through modify_data(),
# which owns the only human-in-the-loop confirmation in this system. The system
# instruction reinforces this at the prompt level; the actual enforcement lives
# in code (modify_data's input() prompt), so even if the model ignored the
# instruction, the guardrail would still hold.
SYSTEM_INSTRUCTION = SKILL_DEFINITION + f"\n\nDataset columns available: {list(df.columns)}"

chat_history = []  # Conversational memory: full turn history, not just last message

def ask_agent(user_message: str):
    """
    Sends a user message to the agent, appends it to the running chat history
    (context engineering: the model sees the whole conversation, enabling
    references like 'that column' to resolve correctly), and lets Gemini
    decide whether to call one of AVAILABLE_TOOLS.
    """
    chat_history.append(types.Content(role="user", parts=[types.Part(text=user_message)]))

    response = client.models.generate_content(
        model=MODEL,
        contents=chat_history,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            tools=[describe_data, detect_outliers, run_correlation, plot_chart, modify_data],
        ),
    )

    print("🤖 DataScout:", response.text)
    print("\n📌 Session memory (accumulated actions):", session_memory["actions_taken"])
    chat_history.append(types.Content(role="model", parts=[types.Part(text=response.text)]))
    return response.text

## 4. Try it out

In [ ]:
ask_agent("Give me a general summary of the dataset")

In [ ]:
ask_agent("Are there outliers in any numeric column? Pick the first numeric column you find.")

In [ ]:
ask_agent("Plot a histogram of that same column")

In [ ]:
# Example of a sensitive action with the human-in-the-loop guardrail
ask_agent("Drop the nulls in that column")

## 5. Notes for the Writeup

- **Innovation**: the agent doesn't follow a fixed pipeline — it dynamically decides
  which tool to use based on the question, and keeps session memory to avoid
  repeating analysis already done.
- **Security**: actions that modify data go through explicit human confirmation
  (human-in-the-loop), following the course's "Effective Trust" framework. A
  STRIDE-style review (run via Antigravity IDE) also identified and fixed an
  argument-validation gap — see `SECURITY.md`.
- **Deployability**: the API key is never exposed in the code — it's managed via
  Colab Secrets.
- **Future extensions**: memory persistence across sessions, more tools (statistical
  tests, PDF/Excel export), multi-agent setups (a cleaning agent + an analysis agent).